# Nobel Atlas: проверка в Google Colab

Сначала загрузите исправленные файлы в ваш репозиторий. Выполняйте ячейки сверху вниз.
Здесь используется отдельная среда Python и управляемый временный туннель Cloudflare.
Ссылка общедоступна и работает только пока запущены процессы и среда Colab. Для постоянного сайта используйте Streamlit Community Cloud.
Последнюю ячейку остановки запускайте только после работы с сайтом, не вместе с остальными.


In [ ]:
from pathlib import Path
import subprocess
import sys

# Повторный запуск этой ячейки сначала останавливает наши старые процессы.
if "preview" in globals():
    preview.stop()

REPO = Path("/content/nobel_atlas")
if not REPO.exists():
    subprocess.run(["git", "clone", "https://github.com/Denbrel/nobel_atlas.git", str(REPO)], check=True)
else:
    dirty = subprocess.check_output(["git", "-C", str(REPO), "status", "--porcelain"], text=True)
    if dirty.strip():
        raise RuntimeError("В копии есть локальные изменения. Сохраните их перед обновлением репозитория.")
    subprocess.run(["git", "-C", str(REPO), "pull", "--ff-only"], check=True)

if not (REPO / "scripts/colab_preview.py").exists():
    raise RuntimeError("Сначала загрузите пакет исправлений в GitHub и повторите эту ячейку.")

ENV = Path("/content/nobel_atlas_env")
if not (ENV / "bin/python").exists():
    result = subprocess.run([sys.executable, "-m", "venv", str(ENV)])
    if result.returncode:
        subprocess.run([sys.executable, "-m", "pip", "install", "virtualenv"], check=True)
        subprocess.run([sys.executable, "-m", "virtualenv", str(ENV)], check=True)
PYTHON = ENV / "bin/python"
subprocess.run([str(PYTHON), "-m", "pip", "install", "-r", str(REPO / "requirements.txt")], check=True)
subprocess.run([str(PYTHON), "-m", "pip", "check"], check=True)
subprocess.run([str(PYTHON), str(REPO / "scripts/check_deployment.py")], cwd=REPO, check=True)


## Проверка файлов интерфейса

Проверяет HTTP-выдачу установленных JavaScript/CSS-модулей и карты до создания публичного туннеля. Ошибка на этом шаге сохраняется в выводе.


In [ ]:
subprocess.run([str(PYTHON), str(REPO / "scripts/smoke_http.py")], cwd=REPO, check=True)


## Открыть приложение

Первый запуск скачает официальный `cloudflared` с проверкой SHA-256. Процесс сначала ждёт готовности Streamlit, затем — готовности публичной ссылки. Пароль в виде IP не нужен. CORS и XSRF остаются включёнными.


In [ ]:
import importlib.util
from IPython.display import display, HTML
from html import escape

if "preview" in globals():
    preview.stop()
module_spec = importlib.util.spec_from_file_location("atlas_preview", REPO / "scripts/colab_preview.py")
module = importlib.util.module_from_spec(module_spec)
module_spec.loader.exec_module(module)
preview = module.PreviewRuntime(REPO, python=PYTHON)
url = preview.start()
display(HTML(f'<a href="{escape(url)}" target="_blank" rel="noopener">Открыть Nobel Atlas ↗</a>'))


## Если нужен журнал

Выполните следующую ячейку только для диагностики. Если ссылка перестала работать после отключения среды Colab, снова выполните запуск и откройте **новую** ссылку.


In [ ]:
for name in ("streamlit.log", "tunnel.log"):
    path = REPO / ".preview-runtime" / name
    print(f"\n{name}")
    print(path.read_text(errors="replace")[-8000:] if path.exists() else "Журнал пока не создан")


## Остановить предпросмотр

Эту ячейку выполняйте, когда закончите работу. Она останавливает только процессы этого ноутбука.


In [ ]:
if "preview" in globals():
    preview.stop()
    print("Предпросмотр остановлен.")
